# BLOK 1 & 2

In [ ]:
import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np

# ==========================================
# 1. KONFIGURASI HALAMAN
# ==========================================
st.set_page_config(page_title="Supplier Intelligence DSS", page_icon="🏢", layout="wide")

- Logika Backend: * @st.cache_data: Ini adalah teknik Memoization. Backend menyimpan hasil eksekusi ini di RAM server. Saat pengguna menggeser parameter di dashboard, fungsi berat ini tidak diulang, membuat aplikasi merespons dalam hitungan milidetik.
- Fungsi vektor Pandas (.str.replace, .dt.days) digunakan karena jauh lebih cepat (Big O(1) secara efisiensi komputasi) dibandingkan melakukan looping row-by-row.

Matematika Dasar: Operasi skalar sederhana untuk mendapatkan selisih waktu mutlak: $Shelf\_Life = Expiration - Received$.

In [ ]:
# ==========================================
# 2. FUNGSI CACHE & PREPROCESSING DATA
# ==========================================
@st.cache_data
def load_data():
    df = pd.read_csv("data/supplier_data.csv")

    # Bersihkan kolom teks
    df['Status'] = df['Status'].astype(str).str.strip()
    df['Catagory'] = df['Catagory'].astype(str).str.strip()
    df['Product_Name'] = df['Product_Name'].astype(str).str.strip()
    df['Supplier_Name'] = df['Supplier_Name'].astype(str).str.strip()

    # Bersihkan harga
    df['Unit_Price'] = (
        df['Unit_Price']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )

    # Konversi tanggal
    df['Date_Received'] = pd.to_datetime(df['Date_Received'], errors='coerce')
    df['Expiration_Date'] = pd.to_datetime(df['Expiration_Date'], errors='coerce')

    # Hitung umur simpan
    df['Shelf_Life_Days'] = (df['Expiration_Date'] - df['Date_Received']).dt.days

    # Turnover rate
    df['Turnover_Rate'] = pd.to_numeric(df['Inventory_Turnover_Rate'], errors='coerce')

    # Hapus data yang tidak valid
    df = df.dropna(subset=['Unit_Price', 'Turnover_Rate', 'Shelf_Life_Days'])

    return df


df = load_data()


# ==========================================
# FUNGSI NORMALISASI
# ==========================================
def min_max_benefit(col):
    if col.max() == col.min():
        return np.ones(len(col))
    return (col - col.min()) / (col.max() - col.min())


def min_max_cost(col):
    if col.max() == col.min():
        return np.ones(len(col))
    return (col.max() - col) / (col.max() - col.min())

# BLOK 3

- Domain Knowledge: Harga harus serendah mungkin (Cost), sedangkan Umur Simpan harus selama mungkin (Benefit).
- Logika Backend: Logika kondisional if col.max() != col.min() berfungsi sebagai pengaman. Jika semua pemasok mematok harga yang sama $5, pembaginya akan bernilai $0$ dan membuat program crash. Oleh karena itu, kita paksa nilainya menjadi $1.0$.
- Matematika: Formula Min-Max Scaling. Ini memaksa semua nilai metrik mentah menjadi rentang rasio $0.0$ hingga $1.0$.

In [ ]:
# ==========================================
# 3. SIDEBAR (PANEL KENDALI UTAMA)
# ==========================================
st.sidebar.title("Control Panel")
menu = st.sidebar.radio("Pilih Mode Analisis:", 
    ["1. Evaluasi Aktif", "2. Audit Post-Mortem", "3. Eksplorasi Data Mentah"])

st.sidebar.markdown("---")
st.sidebar.subheader("Simulator Bobot Kriteria")
st.sidebar.caption("Ketik persentase prioritas bisnis (Total harus 100).")

w_price = st.sidebar.number_input("Bobot Harga (Cost) %", min_value=0, max_value=100, value=30, step=1)
w_turnover = st.sidebar.number_input("Bobot Kecepatan Laku (Benefit) %", min_value=0, max_value=100, value=20, step=1)
w_shelf = st.sidebar.number_input("Bobot Kesegaran/Umur (Benefit) %", min_value=0, max_value=100, value=50, step=1)

total_weight = w_price + w_turnover + w_shelf
if total_weight != 100:
    st.sidebar.error(f"Total bobot saat ini {total_weight}%. Total HARUS 100%!")
    st.stop() 

w_price, w_turnover, w_shelf = w_price/100, w_turnover/100, w_shelf/100

# BLOK 4

- Domain Knowledge: Di sinilah Manajer menentukan strategi bisnisnya. Penggunaan Direct Weighting (input langsung) dipilih karena lebih ramah pengguna (User Friendly) dibanding mengisi matriks perbandingan berpasangan AHP yang rumit.
- Logika Backend: Fungsi st.stop() akan secara brutal menghentikan eksekusi kode di bawahnya jika total input manajer lebih atau kurang dari 100%. Ini mencegah kalkulasi "sampah".
- Matematika: Penegakan hukum probabilitas pembobotan mutlak: $\sum w = 1$. Persentase dibagi 100 untuk menjadi pengali desimal.

In [ ]:
# ==========================================
# 4. HALAMAN 1: EVALUASI AKTIF
# ==========================================
if menu == "1. Evaluasi Aktif":
    st.title("Analisa Pengadaan Barang Aktif")
    st.write("Analisis Ensemble untuk mencari pemasok terbaik. *Hanya menampilkan produk berstatus Active & Backordered.*")
    
    df_active = df[df['Status'].isin(['Active', 'Backordered'])].copy()
    
    # --- UI BARU: Pilihan Level Evaluasi ---
    eval_level = st.radio("Pilih Level Evaluasi:", ["Per Produk Spesifik", "Keseluruhan Kategori (Global)"], horizontal=True)
    
    col1, col2 = st.columns(2)
    with col1:
        selected_cat = st.selectbox("1. Pilih Kategori:", options=df_active['Catagory'].dropna().unique())
        
    # --- LOGIKA FILTERING ---
    if eval_level == "Per Produk Spesifik":
        with col2:
            products_in_cat = df_active[df_active['Catagory'] == selected_cat]['Product_Name'].unique()
            selected_prod = st.selectbox("2. Pilih Produk Spesifik:", options=products_in_cat)
        df_eval = df_active[(df_active['Catagory'] == selected_cat) & (df_active['Product_Name'] == selected_prod)].copy()
        target_name = selected_prod
    else:
        # Jika global, abaikan spesifik produk, ambil semua di kategori tersebut
        df_eval = df_active[df_active['Catagory'] == selected_cat].copy()
        target_name = f"Kategori {selected_cat}"
    
    # Validasi jumlah pesaing (menggunakan nunique agar lebih presisi)
    if df_eval['Supplier_Name'].nunique() < 2:
        st.warning(f"Hanya ada {df_eval['Supplier_Name'].nunique()} pemasok untuk {target_name}. Tidak ada persaingan untuk dihitung.")
    else:
        # Agregasi Rata-rata per Supplier
        grouped = df_eval.groupby('Supplier_Name').agg({
            'Unit_Price': 'mean',
            'Turnover_Rate': 'mean',
            'Shelf_Life_Days': 'mean'
        }).reset_index()
        
        # --- KOMPUTASI SAW & TOPSIS ---
        grouped['Norm_Price'] = min_max_cost(grouped['Unit_Price'])
        grouped['Norm_Turnover'] = min_max_benefit(grouped['Turnover_Rate'])
        grouped['Norm_Shelf'] = min_max_benefit(grouped['Shelf_Life_Days'])
        
        # Skor SAW
        grouped['Skor_SAW'] = (grouped['Norm_Price'] * w_price) + (grouped['Norm_Turnover'] * w_turnover) + (grouped['Norm_Shelf'] * w_shelf)
        
        # Skor TOPSIS
        grouped['V_Price'] = grouped['Norm_Price'] * w_price
        grouped['V_Turnover'] = grouped['Norm_Turnover'] * w_turnover
        grouped['V_Shelf'] = grouped['Norm_Shelf'] * w_shelf
        
        A_plus = np.array([w_price, w_turnover, w_shelf])
        A_minus = np.array([0.0, 0.0, 0.0])
        
        def calc_d_plus(row):
            return np.sqrt((row['V_Price'] - A_plus[0])**2 + (row['V_Turnover'] - A_plus[1])**2 + (row['V_Shelf'] - A_plus[2])**2)
            
        def calc_d_minus(row):
            return np.sqrt((row['V_Price'] - A_minus[0])**2 + (row['V_Turnover'] - A_minus[1])**2 + (row['V_Shelf'] - A_minus[2])**2)
            
        grouped['D_Plus'] = grouped.apply(calc_d_plus, axis=1)
        grouped['D_Minus'] = grouped.apply(calc_d_minus, axis=1)
        grouped['Skor_TOPSIS'] = grouped['D_Minus'] / (grouped['D_Plus'] + grouped['D_Minus'])
        
        st.markdown("---")
        
        # --- UI BARU: Pilihan Metode Visualisasi ---
        selected_method = st.radio("Tampilkan Grafik Berdasarkan Metode:", ["TOPSIS", "SAW"], horizontal=True)
        score_col = 'Skor_TOPSIS' if selected_method == "TOPSIS" else 'Skor_SAW'
        
        # Urutkan berdasarkan metode yang dipilih
        grouped = grouped.sort_values(score_col, ascending=False).reset_index(drop=True)
        grouped.index += 1 
        
        top_supplier = grouped.iloc[0]
        st.subheader(f"Rekomendasi untuk {target_name}: {top_supplier['Supplier_Name']}")
        
        # Visualisasi & Tabel
        col_chart, col_table = st.columns([1, 1])
        with col_chart:
            fig = px.bar(grouped, x='Supplier_Name', y=score_col, 
                         title=f"Papan Peringkat ({selected_method})", 
                         color=score_col, color_continuous_scale='plasma')
            st.plotly_chart(fig, use_container_width=True)
            
        with col_table:
            # 1. Tentukan kolom dasar dari backend
            base_cols = ['Supplier_Name', 'Unit_Price', 'Turnover_Rate', 'Shelf_Life_Days']
            display_cols = base_cols + [score_col]
            
            # 2. Kamus (Dictionary) untuk mengubah nama kolom agar cantik (Customer-Facing)
            rename_dict = {
                'Supplier_Name': 'Nama Pemasok',
                'Unit_Price': 'Harga Satuan',
                'Turnover_Rate': 'Kecepatan Laku',
                'Shelf_Life_Days': 'Umur Simpan',
                'Skor_SAW': 'Skor Akhir (SAW)',
                'Skor_TOPSIS': 'Skor Akhir (TOPSIS)'
            }
            
            # 3. Buat DataFrame khusus untuk tampilan & ubah namanya
            df_display = grouped[display_cols].rename(columns=rename_dict)
            
            # 4. Ambil nama kolom skor yang baru sesuai pilihan toggle
            score_col_display = rename_dict[score_col]
            
            # 5. Format teks disesuaikan dengan NAMA KOLOM BARU
            format_dict = {
                'Harga Satuan': '${:.2f}', 
                'Kecepatan Laku': '{:.1f}x', 
                'Umur Simpan': '{:.0f} Hari', 
                score_col_display: '{:.3f}'
            }
            
            # 6. Render tabelnya
            st.dataframe(df_display.style.format(format_dict), use_container_width=True)

# BLOK 5

- Domain Knowledge: Mengevaluasi kinerja satu supplier tidak bisa dari satu transaksi terbaik/terburuknya saja. Kita harus mencari performa wajar (Expected Value) dari seluruh riwayat pengirimannya.
- Logika Backend:
    - Penggunaan groupby().agg('mean') akan memeras (merangkum) ribuan baris transaksi per nama supplier menjadi satu baris berisi nilai rata-ratanya.
    - Penggunaan st.radio (Toggle SAW/TOPSIS) di bagian UI memungkinkan tabel berubah secara dinamis berdasarkan perhitungan yang dipilih.
- Matematika:
    - SAW: Penjumlahan linier perkalian rasio dengan bobot $\rightarrow V_i = \sum w_j \cdot r_{ij}$.
    - TOPSIS: Pendekatan geometri ruang. Menghitung jarak (Euclidean Distance) pemasok dari titik paling ideal ($A^+$) dan titik paling hancur ($A^-$). Kandidat pemenang harus sangat dekat dengan $A^+$ dan sangat jauh dari $A^-$

In [ ]:
# ==========================================
# 5. HALAMAN 2: AUDIT POST-MORTEM
# ==========================================
elif menu == "2. Audit Post-Mortem":

    st.title("Audit Kinerja Pemasok (Pasca Discontinued)")
    st.write(
        "Analisis arsip data masa lalu untuk mencari pola kegagalan pemasok. "
        "*Hanya menampilkan produk berstatus Discontinued.*"
    )

    # Filter produk discontinued
    df_disc = df[df["Status"] == "Discontinued"].copy()

    # Red flag jika umur simpan negatif
    df_disc["Red_Flag"] = df_disc["Shelf_Life_Days"] < 0

    # ==================================
    # NORMALISASI PER KATEGORI
    # ==================================
    def normalize_audit(group):
        group["Norm_Price"] = min_max_cost(group["Unit_Price"])
        group["Norm_Turn"] = min_max_benefit(group["Turnover_Rate"])
        group["Norm_Shelf"] = min_max_benefit(group["Shelf_Life_Days"])
        return group

    df_disc = df_disc.groupby("Catagory", group_keys=False).apply(normalize_audit)

    # ==================================
    # HITUNG SKOR SAW
    # ==================================
    df_disc["Skor_SAW"] = (
        (df_disc["Norm_Price"] * w_price)
        + (df_disc["Norm_Turn"] * w_turnover)
        + (df_disc["Norm_Shelf"] * w_shelf)
    )

    st.markdown("---")

    col_gem, col_black = st.columns(2)

    # ==================================
    # HIDDEN GEMS
    # ==================================
    with col_gem:

        st.subheader("SUPPLIER HIDDEN GEMS")
        st.caption(
            "Barang bagus (tanpa catatan merah) yang salah dihentikan. "
            "**Rekomendasi: Negosiasi Ulang!**"
        )

        gems = (
            df_disc[~df_disc["Red_Flag"]]
            .sort_values("Skor_SAW", ascending=False)
            .head(10)
        )

        gems_display = gems[
            ["Supplier_Name", "Product_Name", "Catagory", "Skor_SAW"]
        ].rename(
            columns={
                "Supplier_Name": "Nama Pemasok",
                "Product_Name": "Nama Produk",
                "Catagory": "Kategori",
                "Skor_SAW": "Skor Performa",
            }
        )

        st.dataframe(
            gems_display.style.format({"Skor Performa": "{:.3f}"}),
            use_container_width=True,
        )

    # ==================================
    # BLACKLIST SUPPLIER
    # ==================================
    with col_black:

        st.subheader("SUPPLIER BLACKLIST")
        st.caption(
            "Pemasok dengan riwayat paling banyak mengirim barang busuk (Red Flag). "
            "**Rekomendasi: Blokir!**"
        )

        blacklist = (
            df_disc.groupby("Supplier_Name")
            .agg(
                Total_Gagal=("Product_ID", "count"),
                Total_Red_Flags=("Red_Flag", "sum"),
                Rata_Skor=("Skor_SAW", "mean"),
            )
            .reset_index()
            .sort_values(["Total_Red_Flags", "Rata_Skor"], ascending=[False, True])
            .head(10)
        )

        blacklist_display = blacklist.rename(
            columns={
                "Supplier_Name": "Nama Pemasok",
                "Total_Gagal": "Total Produk Gagal",
                "Total_Red_Flags": "Total Pelanggaran (Kedaluwarsa)",
                "Rata_Skor": "Rata-rata Skor",
            }
        )

        st.dataframe(
            blacklist_display.style.format({"Rata-rata Skor": "{:.3f}"}),
            use_container_width=True,
        )

# BLOK 6

- Domain Knowledge: Manajemen Risiko. Pemasok yang mengirim barang kedaluwarsa (umur simpan negatif) harus di-flag (ditandai). Jika hal ini terjadi berulang kali, mereka masuk daftar hitam (Blacklist).
- Logika Backend:
    - .groupby('Catagory').apply() memastikan normalisasi terisolasi. Ini penting karena secara statistik, umur maksimal daging tidak bisa dibandingkan dengan umur maksimal beras. Mereka harus dinormalisasi dengan saingan di industrinya sendiri.
- Matematika:
    - Variabel Boolean Red_Flag (True/False). Penggunaan .sum() pada kumpulan data Boolean secara otomatis menghitung frekuensi kejadian (berapa kali 'True' muncul).
    - Operator bitwise ~ (NOT) digunakan untuk menarik produk Hidden Gems yang syaratnya mutlak bernilai False pada kolom Red_Flag (berkualitas prima tanpa

In [ ]:
# ==========================================
# 6. HALAMAN 3: RAW DATA EKSPLORASI
# ==========================================
else:
    st.title("Eksplorasi Data Mentah")
    st.write("Tampilan *database* utuh untuk keperluan audit manual tim IT.")

    # KPI SUMMARY
    col_kpi1, col_kpi2, col_kpi3 = st.columns(3)

    with col_kpi1:
        st.metric("Total Transaksi", len(df))

    with col_kpi2:
        st.metric("Total Pemasok", df['Supplier_Name'].nunique())

    with col_kpi3:
        st.metric("Total Produk", df['Product_Name'].nunique())

    st.markdown("---")

    # SEARCH BOX
    search = st.text_input("Cari Nama Pemasok atau Produk:")

    if search:
        filtered_df = df[
            df['Supplier_Name'].astype(str).str.contains(search, case=False, na=False) |
            df['Product_Name'].astype(str).str.contains(search, case=False, na=False)
        ]

        st.dataframe(filtered_df, use_container_width=True)

    else:
        st.dataframe(df, use_container_width=True)